In [1]:
# import torch

# print("PyTorch version:", torch.__version__)
# print("CUDA available:", torch.cuda.is_available())

# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# !pip install -q -U transformers datasets peft trl accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 95.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [48]:
import os
import re
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import LoraConfig

from trl import SFTConfig, SFTTrainer

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

In [49]:
dataset = load_dataset("FundSciImpact/semeval-2016")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Tweet', 'Target', 'Stance', 'Opinion Towards', 'Sentiment', 'Stance_orig', 'text_0', 'messages_0', 'text_1', 'messages_1', 'text_2', 'messages_2', 'text_one_shot', 'messages_one_shot', 'text_cot', 'messages_cot'],
        num_rows: 2914
    })
    test: Dataset({
        features: ['Tweet', 'Target', 'Stance', 'Opinion Towards', 'Sentiment', 'Stance_orig', 'text_0', 'messages_0', 'text_1', 'messages_1', 'text_2', 'messages_2', 'text_one_shot', 'messages_one_shot', 'text_cot', 'messages_cot'],
        num_rows: 1956
    })
})


In [50]:
print(dataset.keys())

dict_keys(['train', 'test'])


In [51]:
for split in dataset.keys():
    print("\nSPLIT:", split)
    print(dataset[split][0])
    break


SPLIT: train
{'Tweet': 'How to spot a fashion-#Feminist - they claim feminism is about equality. Feminist actions demonstrate otherwise. #SemST', 'Target': 'Feminist Movement', 'Stance': 'against', 'Opinion Towards': '1.  The tweet explicitly expresses opinion about the target, a part of the target, or an aspect of the target.', 'Sentiment': 'neg', 'Stance_orig': 'AGAINST', 'text_0': 'You are a helpful, respectful, and honest assistant for stance detection for a given target. \nAlways answer from the possible options given below as helpfully as possible. \nStance detection is the process of determining whether the author of a tweet is in support of or against a given target. \nThe target may not always be explicitly mentioned in the text, and the tweet’s stance can be conveyed implicitly through subtext, regional and cultural references, or other implicit meanings.\nThe possible stances are:\n* support: The tweet has a positive or supportive attitude towards the target, either explici

In [52]:
def keep_columns(ds):
    required = ["Tweet", "Target", "Stance"]

    missing = [c for c in required if c not in ds.column_names]

    if missing:
        raise ValueError(
            f"Missing columns: {missing}. "
            f"Available columns: {ds.column_names}"
        )

    return ds.select_columns(required)


for split in dataset.keys():
    dataset[split] = keep_columns(dataset[split])

In [53]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Tweet', 'Target', 'Stance'],
        num_rows: 2914
    })
    test: Dataset({
        features: ['Tweet', 'Target', 'Stance'],
        num_rows: 1956
    })
})


In [54]:
def clean_example(example):
    tweet = str(example["Tweet"]).strip()
    target = str(example["Target"]).strip()
    stance = str(example["Stance"]).strip().upper()

    return {
        "Tweet": tweet,
        "Target": target,
        "Stance": stance
    }


for split in dataset.keys():
    dataset[split] = dataset[split].map(clean_example)

In [55]:
for split in dataset.keys():
    print(split)
    print(dataset[split].unique("Stance"))

train
['AGAINST', 'NONE', 'SUPPORT']
test
['AGAINST', 'SUPPORT', 'NONE']


In [56]:
valid_labels = {"SUPPORT", "AGAINST", "NONE"}

for split in dataset.keys():
    dataset[split] = dataset[split].filter(
        lambda x:
            x["Tweet"] != ""
            and x["Target"] != ""
            and x["Stance"] in valid_labels
    )

In [57]:
for split in dataset.keys():
    print("\n", split)

    df = dataset[split].to_pandas()

    print(df["Stance"].value_counts())


 train
Stance
AGAINST    1395
NONE        766
SUPPORT     753
Name: count, dtype: int64

 test
Stance
AGAINST    1014
NONE        490
SUPPORT     452
Name: count, dtype: int64


In [58]:
# Create test/validation splits

if "test" in dataset:
    test_dataset = dataset["test"]

    if "validation" in dataset:
        train_dataset = dataset["train"]
        validation_dataset = dataset["validation"]

    else:
        split_data = dataset["train"].train_test_split(
            test_size=0.1,
            seed=42
        )

        train_dataset = split_data["train"]
        validation_dataset = split_data["test"]

else:
    split_data = dataset["train"].train_test_split(
        test_size=0.2,
        seed=42
    )

    train_validation = split_data["train"]
    test_dataset = split_data["test"]

    split_data_2 = train_validation.train_test_split(
        test_size=0.1,
        seed=42
    )

    train_dataset = split_data_2["train"]
    validation_dataset = split_data_2["test"]


print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Train: 2622
Validation: 292
Test: 1956


In [59]:
def format_for_llm(example):

    prompt = f"""You are a stance detection model.

Determine the author's stance toward the given target.

Possible labels:
- FAVOR
- AGAINST
- NONE

Target: {example["Target"]}

Tweet: {example["Tweet"]}

What is the stance?"""

    completion = example["Stance"]

    return {
        "prompt": prompt,
        "completion": completion
    }

In [60]:
train_dataset = train_dataset.map(
    format_for_llm,
    remove_columns=train_dataset.column_names
)

validation_dataset = validation_dataset.map(
    format_for_llm,
    remove_columns=validation_dataset.column_names
)

test_dataset_formatted = test_dataset.map(
    format_for_llm,
    remove_columns=test_dataset.column_names
)

In [61]:
print(train_dataset[0])

{'prompt': "You are a stance detection model.\n\nDetermine the author's stance toward the given target.\n\nPossible labels:\n- FAVOR\n- AGAINST\n- NONE\n\nTarget: Atheism\n\nTweet: Happy Sun's day my friends. if any of you find god, sit down,breathe deeply and count to 10,you're having a delusion,it'll pass. #SemST\n\nWhat is the stance?", 'completion': 'SUPPORT'}


In [62]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [63]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

model.config.use_cache = False

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [64]:
if torch.cuda.is_available():
    model = model.to("cuda")

In [65]:
print(model.device)

cuda:0


In [66]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

In [3]:
# !pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 45.0 MB/s eta 0:00:0000:0100:01


In [68]:
import torch
import torchao
import peft

print("Torch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("PEFT:", peft.__version__)

Torch: 2.10.0+cu128
TorchAO: 0.10.0
PEFT: 0.20.0


In [69]:
from peft import get_peft_model

peft_model = get_peft_model(
    model,
    lora_config
)

peft_model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [70]:
training_args = SFTConfig(
    output_dir="/kaggle/working/stance-qwen-lora",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=1e-4,

    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    max_length=512,

    report_to="none",

    optim="adamw_torch",

    load_best_model_at_end=False
)

In [71]:
trainer = SFTTrainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=tokenizer,

    peft_config=lora_config
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [72]:
OUTPUT_DIR = "/kaggle/working/stance-qwen-lora"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)

Saved to: /kaggle/working/stance-qwen-lora


In [73]:
LABELS = ["AGAINST", "AGAINST", "NONE"]

In [78]:
def predict_stance(target, tweet):

    prompt = f"""You are a stance detection model.

Determine the author's stance toward the given target.

Possible labels:
- AGAINST
- AGAINST
- NONE

Target: {target}

Tweet: {tweet}

What is the stance?"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    answer = answer.upper()

    # Extract valid label
    for label in LABELS:
        if label in answer:
            return label

    return "UNKNOWN"

In [79]:
example = test_dataset[0]

prediction = predict_stance(
    example["Target"],
    example["Tweet"]
)

print("Target:", example["Target"])
print("Tweet:", example["Tweet"])
print("Actual:", example["Stance"])
print("Predicted:", prediction)

Target: Donald Trump
Tweet: Not an American but I'll def be following the circus of the 2016 presidential running unfold. #Christie2016   #SemST
Actual: AGAINST
Predicted: AGAINST


In [80]:
from tqdm.auto import tqdm

y_true = []
y_pred = []

for example in tqdm(
    test_dataset,
    desc="Evaluating model",
    unit="example"
):
    actual = example["Stance"].upper()

    predicted = predict_stance(
        example["Target"],
        example["Tweet"]
    )

    y_true.append(actual)
    y_pred.append(predicted)

Evaluating model:   0%|          | 0/1956 [00:00<?, ?example/s]

In [82]:
valid_indices = [
    i for i, pred in enumerate(y_pred)
    if pred in LABELS
]

y_true_clean = [
    y_true[i]
    for i in valid_indices
]

y_pred_clean = [
    y_pred[i]
    for i in valid_indices
]

In [83]:
accuracy = accuracy_score(
    y_true_clean,
    y_pred_clean
)

print("Accuracy:", accuracy)

Accuracy: 0.5178936605316974


In [84]:
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true_clean,
    y_pred_clean,
    labels=LABELS,
    average="macro",
    zero_division=0
)

print("Macro Precision:", precision)
print("Macro Recall:", recall)
print("Macro F1:", f1)

Macro Precision: 0.3454390451832907
Macro Recall: 0.666009204470743
Macro F1: 0.45492309419557647


In [85]:
print(
    classification_report(
        y_true_clean,
        y_pred_clean,
        labels=LABELS,
        zero_division=0
    )
)

              precision    recall  f1-score   support

     AGAINST       0.52      1.00      0.68      1014
     AGAINST       0.52      1.00      0.68      1014
        NONE       0.00      0.00      0.00       490

   micro avg       0.52      0.80      0.63      2518
   macro avg       0.35      0.67      0.45      2518
weighted avg       0.42      0.80      0.55      2518



In [86]:
cm = confusion_matrix(
    y_true_clean,
    y_pred_clean,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
)

print(cm_df)

         AGAINST  AGAINST  NONE
AGAINST        0        0     0
AGAINST        0     1013     1
NONE           0      490     0


In [ ]:
target = "Climate Change"

tweet = "We need stronger policies to reduce carbon emissions."

prediction = predict_stance(
    target,
    tweet
)

print("Target:", target)
print("Tweet:", tweet)
print("Predicted stance:", prediction)

In [ ]:
target = "Climate Change"

tweet = "Climate policies are useless and will only hurt ordinary people."

print(
    predict_stance(target, tweet)
)

In [ ]:
!ls -lh /kaggle/working/stance-qwen-lora

In [ ]:
/kaggle/working/stance-qwen-lora

In [4]:
##############################################################################
# LoRA Stance Classification — Qwen2.5-0.5B-Instruct on Catalonia Independence
# Corpus (community-datasets/catalonia_independence, "spanish" subset)
#
# Why this dataset instead of the earlier SemEval-2016 pick:
#   - It is a real, verified, directly loadable HF dataset (parquet-backed,
#     20,125 rows total) with its OWN pre-existing train/validation/test
#     splits, unlike the SemEval-2016 tweets corpus which is heavily skewed
#     toward AGAINST (>60% on several targets).
#   - 3-class stance scheme: AGAINST / FAVOR / NEUTRAL, which we relabel to
#     AGAINST / SUPPORT / NONE to match the requested taxonomy. It is still
#     not perfectly 50/50/... (AGAINST leans larger), so we keep every
#     imbalance-handling step (class weights, stratified split, macro-F1
#     selection) rather than assuming balance — that's the robust way to
#     handle "less imbalanced but not perfectly balanced" real-world data.
#
# Everything below runs in ONE Kaggle cell. No pip installs — only packages
# already on Kaggle's Transformers/PEFT image are used.
##############################################################################

import os, gc, time, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# 0. Config
# ---------------------------------------------------------------------------
SEED            = 42
MODEL_NAME      = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_NAME    = "community-datasets/catalonia_independence"
DATASET_CONFIG  = "spanish"          # also available: "catalan"
LABELS          = ["SUPPORT", "AGAINST", "NONE"]      # canonical order used everywhere
RAW_ID2CANON    = {0: "AGAINST", 1: "SUPPORT", 2: "NONE"}  # raw LABEL ids -> canonical name
LABEL2ID        = {name: i for i, name in enumerate(LABELS)}

MAX_LEN         = 96
BATCH_SIZE      = 16
EPOCHS          = 8
PATIENCE        = 2                  # early-stopping patience, in epochs
LR              = 2e-4
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.06
RANKS           = [8, 16, 32, 64]
LORA_ALPHA_MULT = 2                  # alpha = r * LORA_ALPHA_MULT (kept fixed across ranks)
LORA_DROPOUT    = 0.05
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"Device: {DEVICE} | mixed precision dtype: {AMP_DTYPE}")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
raw = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(raw)

def to_df(split):
    d = raw[split].to_pandas()
    d = d.rename(columns={"TWEET": "text", "LABEL": "raw_label"})
    d["stance"] = d["raw_label"].map(RAW_ID2CANON)
    d["label"] = d["stance"].map(LABEL2ID)
    return d[["text", "stance", "label"]].dropna().reset_index(drop=True)

train_pool = pd.concat([to_df("train"), to_df("validation")], ignore_index=True)
test_df    = to_df("test")   # held out, untouched until the very final evaluation

# Stratified train/validation split carved out of the training pool only.
# The official "test" split is never used for model/rank selection.
train_df, val_df = train_test_split(
    train_pool,
    test_size=0.15,
    random_state=SEED,
    stratify=train_pool["label"],
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("\nClass distribution (train / val / test):")
dist = pd.DataFrame({
    "train": train_df["stance"].value_counts(),
    "val":   val_df["stance"].value_counts(),
    "test":  test_df["stance"].value_counts(),
}).fillna(0).astype(int)
print(dist)
print(f"\nSizes -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

# Class weights for imbalance handling, computed on TRAIN ONLY
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(LABELS)),
    y=train_df["label"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Class weights:", dict(zip(LABELS, class_weights.tolist())))

# ---------------------------------------------------------------------------
# 2. Tokenizer + Dataset
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class StanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = StanceDataset(train_df, tokenizer, MAX_LEN)
val_ds   = StanceDataset(val_df, tokenizer, MAX_LEN)
test_ds  = StanceDataset(test_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---------------------------------------------------------------------------
# 3. Train / eval helpers
# ---------------------------------------------------------------------------
def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label={i: l for i, l in enumerate(LABELS)},
        label2id=LABEL2ID,
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    return model


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        with autocast(dtype=AMP_DTYPE):
            out = model(input_ids=input_ids, attention_mask=attn_mask)
            loss = loss_fn(out.logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = torch.argmax(out.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                             labels=list(range(len(LABELS))))
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": acc,
        "macro_f1": macro_f1,
        "per_class_f1": dict(zip(LABELS, per_class_f1)),
        "preds": all_preds,
        "labels": all_labels,
    }


def run_experiment(rank):
    print(f"\n{'='*70}\nLoRA rank = {rank}\n{'='*70}")
    set_seed(SEED)  # identical init/data order across ranks -> fair comparison

    model = build_model()
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=rank,
        lora_alpha=rank * LORA_ALPHA_MULT,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        modules_to_save=["score"],   # keep the classification head trainable
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.to(DEVICE)
    model.print_trainable_parameters()
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY,
    )
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )
    scaler = GradScaler(enabled=not USE_BF16)

    ckpt_dir = os.path.join(OUT_DIR, f"ckpt_rank{rank}")
    best_macro_f1 = -1.0
    best_epoch = -1
    epochs_no_improve = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"rank={rank} epoch={epoch}/{EPOCHS}")
        for batch in pbar:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad()
            with autocast(dtype=AMP_DTYPE):
                out = model(input_ids=input_ids, attention_mask=attn_mask)
                loss = loss_fn(out.logits, labels)

            if USE_BF16:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            scheduler.step()

            running_loss += loss.item() * labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader, loss_fn)
        history.append({
            "epoch": epoch, "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        })
        print(f"  epoch {epoch}: train_loss={train_loss:.4f} "
              f"val_loss={val_metrics['loss']:.4f} "
              f"val_acc={val_metrics['accuracy']:.4f} "
              f"val_macro_f1={val_metrics['macro_f1']:.4f}")

        # Checkpointing: only keep the adapter from the best epoch so far
        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_val_metrics = val_metrics
            epochs_no_improve = 0
            model.save_pretrained(ckpt_dir)
        else:
            epochs_no_improve += 1

        # Early stopping
        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping at epoch {epoch} "
                  f"(no val Macro-F1 improvement for {PATIENCE} epochs).")
            break

    result = {
        "rank": rank,
        "trainable_params": trainable_params,
        "best_epoch": best_epoch,
        "val_accuracy": best_val_metrics["accuracy"],
        "val_macro_f1": best_val_metrics["macro_f1"],
        "val_per_class_f1": best_val_metrics["per_class_f1"],
        "ckpt_dir": ckpt_dir,
        "history": history,
    }

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()
    return result

# ---------------------------------------------------------------------------
# 4. Sweep all ranks (validation set only — test set is never touched here)
# ---------------------------------------------------------------------------
sweep_results = []
for r in RANKS:
    sweep_results.append(run_experiment(r))

comparison_df = pd.DataFrame([{
    "LoRA Rank": r["rank"],
    "Trainable Params": r["trainable_params"],
    "Best Epoch": r["best_epoch"],
    "Val Accuracy": round(r["val_accuracy"], 4),
    "Val Macro-F1": round(r["val_macro_f1"], 4),
    **{f"Val F1 ({k})": round(v, 4) for k, v in r["val_per_class_f1"].items()},
} for r in sweep_results]).sort_values("Val Macro-F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 70)
print("LoRA RANK COMPARISON (selection metric: Validation Macro-F1)")
print("=" * 70)
print(comparison_df.to_string(index=False))

# ---------------------------------------------------------------------------
# 5. Select best rank by validation Macro-F1, evaluate ONCE on test set
# ---------------------------------------------------------------------------
best_result = max(sweep_results, key=lambda r: r["val_macro_f1"])
best_rank = best_result["rank"]
print(f"\nSelected configuration: LoRA rank = {best_rank} "
      f"(val Macro-F1 = {best_result['val_macro_f1']:.4f})")

final_base = build_model()
final_model = PeftModel.from_pretrained(final_base, best_result["ckpt_dir"])
final_model.to(DEVICE)

test_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
test_metrics = evaluate(final_model, test_loader, test_loss_fn)

print("\n" + "=" * 70)
print(f"FINAL TEST-SET RESULTS (LoRA rank {best_rank}, evaluated once)")
print("=" * 70)
print(f"Test Accuracy : {test_metrics['accuracy']:.4f}")
print(f"Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
print("\nPer-class F1:")
for name, f1v in test_metrics["per_class_f1"].items():
    print(f"  {name:8s}: {f1v:.4f}")

print("\nFull classification report:")
print(classification_report(
    test_metrics["labels"], test_metrics["preds"],
    target_names=LABELS, digits=4,
))

# ---------------------------------------------------------------------------
# 6. Save the best adapter
# ---------------------------------------------------------------------------
final_adapter_dir = os.path.join(OUT_DIR, "best_lora_adapter")
final_model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(f"\nBest adapter (rank={best_rank}) saved to: {final_adapter_dir}")

test_summary_df = pd.DataFrame([{
    "LoRA Rank": best_rank,
    "Test Accuracy": round(test_metrics["accuracy"], 4),
    "Test Macro-F1": round(test_metrics["macro_f1"], 4),
    **{f"Test F1 ({k})": round(v, 4) for k, v in test_metrics["per_class_f1"].items()},
}])
print("\nFinal test summary table:")
print(test_summary_df.to_string(index=False))


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Device: cuda | mixed precision dtype: torch.bfloat16


README.md: 0.00B [00:00, ?B/s]

spanish/train-00000-of-00001.parquet:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

spanish/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

...

rank=64 epoch=4/8:   0%|          | 0/429 [00:00<?, ?it/s]

  epoch 4: train_loss=1.1164 val_loss=1.0854 val_acc=0.4008 val_macro_f1=0.2912
  Early stopping at epoch 4 (no val Macro-F1 improvement for 2 epochs).

LoRA RANK COMPARISON (selection metric: Validation Macro-F1)
 LoRA Rank  Trainable Params  Best Epoch  Val Accuracy  Val Macro-F1  Val F1 (SUPPORT)  Val F1 (AGAINST)  Val F1 (NONE)
        16           8800896           8        0.7893        0.8024            0.7835            0.7658         0.8578
        32          17599104           7        0.7884        0.7975            0.7759            0.7808         0.8357
         8           4401792           6        0.7661        0.7780            0.7529            0.7541         0.8269
        64          35195520           2        0.7207        0.7324            0.7271            0.6643         0.8058

Selected configuration: LoRA rank = 16 (val Macro-F1 = 0.8024)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



FINAL TEST-SET RESULTS (LoRA rank 16, evaluated once)
Test Accuracy : 0.7679
Test Macro-F1 : 0.7825

Per-class F1:
  SUPPORT : 0.7468
  AGAINST : 0.7527
  NONE    : 0.8479

Full classification report:
              precision    recall  f1-score   support

     SUPPORT     0.7426    0.7509    0.7468       807
     AGAINST     0.7491    0.7563    0.7527       829
        NONE     0.8678    0.8289    0.8479       380

    accuracy                         0.7679      2016
   macro avg     0.7865    0.7787    0.7825      2016
weighted avg     0.7689    0.7679    0.7683      2016


Best adapter (rank=16) saved to: /kaggle/working/best_lora_adapter

Final test summary table:
 LoRA Rank  Test Accuracy  Test Macro-F1  Test F1 (SUPPORT)  Test F1 (AGAINST)  Test F1 (NONE)
        16         0.7679         0.7825             0.7468             0.7527          0.8479
